In [3]:
import cv2
import easyocr
import numpy as np
from ultralytics import YOLO

# 1. 모델 및 OCR 엔진 로드
model = YOLO(r'C:\미니프젝\미니프로젝트\번호판인식\runs\detect\korea_plate_GPU_s\weights\best.pt')
reader = easyocr.Reader(['ko', 'en'], gpu=True) # 한글/영어 엔진 로드

# 2. OCR 직전 이진화 처리 (이게 없으면 한글 인식이 매우 힘듭니다)
gray = cv2.cvtColor(plate_roi, cv2.COLOR_BGR2GRAY)
# 적응형 이진화 적용
thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
ocr_result = reader.readtext(thresh)

# 2. 테스트할 파일 경로 (사용자님 경로 사용)
video_path = r'C:\미니프젝\미니프로젝트\Korea Car License Plate.v2i.yolov11\test\images\46_jpg.rf.39bc87def710ee3be3ee4605330b8766.jpg'
cap = cv2.VideoCapture(video_path) 

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 고해상도로 추론하여 작은 번호판 감지율 향상
    results = model(frame, imgsz=1280, conf=0.5)[0]

    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = box.conf[0]
        
        # --- 번호판 글자 인식 과정 ---
        # 1. 번호판 영역 자르기
        plate_roi = frame[y1:y2, x1:x2]
        
        if plate_roi.size != 0:
            # 2. 이미지 전처리: 2배 확대 및 선명화로 인식률 제고
            plate_roi = cv2.resize(plate_roi, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
            
            # 3. EasyOCR 실행
            ocr_result = reader.readtext(plate_roi)
            plate_text = ""
            for (bbox, text, prob) in ocr_result:
                # 신뢰도가 낮은 글자는 제외하거나 합칠 수 있습니다
                plate_text += text + " "
            
            # 4. 화면 출력 (박스와 글자)
            color = (0, 255, 0)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # 글자가 박스 위에 깔끔하게 나오도록 표시
            display_str = f"{plate_text.strip()} ({conf:.2f})"
            cv2.putText(frame, display_str, (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.imshow('Final Plate Recognition', frame)
    # 사진 한 장일 경우 0을 입력해 멈춰서 확인, 영상일 경우 1~100 입력
    if cv2.waitKey(0) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()


0: 1280x1280 1 LicensePlate, 9.9ms
Speed: 56.8ms preprocess, 9.9ms inference, 0.9ms postprocess per image at shape (1, 3, 1280, 1280)
